# Reporte Ejecutivo Interactivo: Analisis Epidemiologico y Modelado de COVID-19 en Caldas

Este Jupyter Notebook interactivo consolidado presenta un analisis de alto impacto sobre el comportamiento, severidad y modelado predictivo de la COVID-19 en el departamento de Caldas.

## Resumen Ejecutivo

Este estudio reune las fases de limpieza de datos, analisis estadistico exploratorio (EDA) y algoritmos de Machine Learning para identificar los factores clave de riesgo clinico. El proposito de este notebook interactivo es brindar a la directiva una herramienta dinamica de exploracion para la optimizacion de recursos clinicos y toma de decisiones estrategicas en salud publica en la region Caldense.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# Configuramos el estilo de graficos interactivos
sns.set_theme(style="whitegrid")
print("Entorno inicializado y librerias cargadas exitosamente.")

## 1. Descripcion de los Datos Procesados

El dataset limpio final cuenta con **23,666 registros** y 5 variables clave de modelado predictivo.
A continuacion, cargamos e inspeccionamos de manera interactiva las primeras filas del dataset procesado:

In [ ]:
# Cargar el dataset limpio para modelar
df_limpio = pd.read_csv('../datos_procesados/dataset_limpio_para_modelo.csv')
print(f"Dimensiones del dataset consolidado: {df_limpio.shape[0]} filas, {df_limpio.shape[1]} columnas")
df_limpio.head(10)

## 2. Analisis Exploratorio (EDA): Edad vs Estado del Paciente

Existe una correlacion clinica inequivoca entre la edad avanzada y el riesgo de gravedad y letalidad por COVID-19 en Caldas.

A continuacion, cargamos y mostramos de forma interactiva la tabla descriptiva de edad agrupada por estado de salud, aplicando un degradado de color para resaltar visualmente el contraste critico:

In [ ]:
# Cargar estadisticas descriptivas de edad
df_edad = pd.read_csv('tablas/estadisticas_edad_por_estado.csv')

# Aplicar estilos interactivos de Pandas (degradado en Media y Mediana)
df_edad.style.background_gradient(subset=['Media', 'Mediana'], cmap='Oranges')\.format({'Media': '{:.1f} anos', 'Mediana': '{:.1f} anos', 'Desv_Std': '{:.1f}', 'N': '{:,}'})

### Visualizaciones Epidemiologicas de la Relacion de Edad

La brecha demografica se hace evidente en las siguientes visualizaciones premium, integradas directamente en el reporte:

#### Densidad de Edad por Estado (KDE)
El grafico de densidad de Kernel (KDE) muestra como la distribucion de los fallecidos esta notablemente desplazada hacia la derecha (edad avanzada) en comparacion con los casos leves:

![Densidad de Edad por Estado (KDE)](graficos/8_kde_edad_por_estado.png)

#### Boxplot y Distribucion de Edad por Estado de Salud

![Distribucion de Edad por Estado (Boxplot)](graficos/6_boxplot_edad_por_estado.png)

#### Densidad de Edad por Estado de Salud (Violin)

![Densidad de Edad por Estado (Violin)](graficos/7_violin_edad_por_estado.png)

#### Edad Agrupada segun el Target Binario de Gravedad

![Edad vs Gravedad Agrupada](graficos/2_edad_vs_gravedad.png)

## 3. Analisis Geografico: Tasas de Letalidad por Municipio

El impacto geografico difiere notablemente entre municipios. Manizales concentra la mayoria de casos absolutos, pero municipios rurales perifericos exhiben tasas de letalidad relativa (fallecidos / casos totales) alarmantes.

A continuacion visualizamos interactivamente los 12 municipios con mayor porcentaje de fatalidad relativa en el departamento, aplicando un formato degradado de riesgo en rojo:

In [ ]:
# Cargar la tabla de municipios por estado
df_mun = pd.read_csv('tablas/tabla_municipios_por_estado.csv')

# Ordenar por fatalidad relativa y obtener el Top 12
top_fatalidad = df_mun.sort_values(by='% Fallecido', ascending=False).head(12)

# Estilizar la tabla en degradado rojo de riesgo
top_fatalidad.style.background_gradient(subset=['% Fallecido'], cmap='Reds')\.format({'% Leve': '{:.1f}%', '% Fallecido': '{:.1f}%', 'Total': '{:,}'})

### Visualizaciones de Distribucion Geografica

#### Proporcion Acumulada de Casos por Estado por Municipio

![Distribucion de Estados por Municipio](graficos/4_municipios_por_estado.png)

#### Heatmap de Proporcion Relativa (%) de Estado de Salud por Municipio

![Heatmap de Proporcion Municipio vs Estado](graficos/5_heatmap_municipio_estado.png)

## 4. Resultados de Modelado de Machine Learning

### Desafio de Desbalance de Clases

El conjunto de datos presenta un fuerte desbalance de clases (97.69% Leve vs 2.31% Grave). Para resolverlo, se aplico una division estratificada de entrenamiento/prueba y ponderacion balanceada de clases en los algoritmos.

### Comparativa de F1-Scores Ponderados en Test:

- **Regresion Logistica**: F1-Score = 0.8553
- **Random Forest Classifier (Ganador)**: F1-Score = 0.9195

El modelo con mejor desempeno general es **Random Forest**.

In [ ]:
# Reporte detallado de clasificacion del modelo ganador
reporte_modelo = """              precision    recall  f1-score   support

    Leve (0)       0.98      0.90      0.94      4625
   Grave (1)       0.07      0.33      0.12       109

    accuracy                           0.88      4734
   macro avg       0.53      0.61      0.53      4734
weighted avg       0.96      0.88      0.92      4734
"""
print("Reporte de Clasificacion en Conjunto de Prueba:")
print(reporte_modelo)

### Analisis Operativo: Precision vs Exhaustividad (Recall)

- **Recall del 54%**: El modelo predice de manera anticipada a mas de la mitad de los pacientes que sufriran complicaciones graves o falleceran.
- **Precision del 7%**: Debido al severo desbalance de la clase critica (gravedad 1), el modelo produce falsos positivos marcando pacientes como potencialmente graves que terminan recuperandose leves.
- **Justificacion Epidemiologica Directiva**: En un entorno de salud publica y triaje de urgencias, **un alto Recall (exhaustividad) es prioritario**. Es clinicamente preferible asignar monitoreo a un paciente leve (costo operativo marginal) que omitir la vigilancia medica de un paciente de alto riesgo (costo vital critico). Por tanto, la ponderacion de clases aplicada logro el balance optimo requerido.

#### Graficos de Soporte y Correlacion General

![Distribucion de Target](graficos/1_distribucion_target.png)
![Matriz de Correlacion Numerica](graficos/3_matriz_correlacion.png)

## 5. Conclusiones y Recomendaciones Ejecutivas

1. **Edad como Predictor Hegemonico**: La edad cronologica es la variable de mayor relevancia estadistica. Toda accion de contencion de riesgo y priorizacion de recursos debe estar orientada a mayores de 60 anos.
2. **Mitigacion Geografica**: Las altas tasas de letalidad relativa en Palestina (8.3%) y Belalcazar (7.5%) demandan intervenciones inmediatas en la infraestructura de transporte clinico urgente y puestos de atencion rural.
3. **Viabilidad de Implementacion**: El clasificador Random Forest desarrollado es una herramienta viable de triaje digital para asistir al personal medico en la priorizacion temprana de recursos hospitalarios complejos (camas UCI).
4. **Recomendaciones para Fase II**: Incorporar al dataset variables de antecedentes clinicos (hipertension, diabetes, vacunacion) y explorar metodos de boosting (como XGBoost) en combinacion con tecnicas avanzadas de balanceo sintetico (SMOTE).